In [4]:
import pandas as pd
delivery_df = pd.read_csv("../data/delivery_locations.csv")

import gurobipy as gp

num_buckets = 8
time_limit = 30  # minutes

for resto in delivery_df['restaurant_chosen'].unique():
    df = delivery_df[delivery_df['restaurant_chosen'] == resto].reset_index()

    n_orders_bucket =  (len(df) // num_buckets) + 1
    n_possible_drones = n_orders_bucket

    df = delivery_df.iloc[:n_orders_bucket].reset_index(drop=True)
    service_time = df["drone_unavailability_time_min"].tolist()

    m = gp.Model("drone_delivery")
    m.setParam("OutputFlag", 0)


    # x[i, d] = 1 if order i is assigned to drone d
    x = m.addVars(n_orders_bucket, n_possible_drones,
                vtype=gp.GRB.BINARY, name="x")

    # y[d] = 1 if drone d is used
    y = m.addVars(n_possible_drones,
                vtype=gp.GRB.BINARY, name="y")

    # Each order is assigned to exactly one drone
    for i in range(n_orders_bucket):
        m.addConstr(
            gp.quicksum(x[i, d] for d in range(n_possible_drones)) == 1
        )

    # A drone may only receive orders if it is activated
    for i in range(n_orders_bucket):
        for d in range(n_possible_drones):
            m.addConstr(x[i, d] <= y[d])

    # Total occupation time of each used drone is at most 30 minutes
    for d in range(n_possible_drones):
        m.addConstr(
            gp.quicksum(x[i, d] * service_time[i] for i in range(n_orders_bucket))
            <= time_limit
        )

    # Minimize required fleet size
    m.setObjective(gp.quicksum(y[d] for d in range(n_possible_drones)),
                gp.GRB.MINIMIZE)

    m.optimize()

    print(f"Restaurant : {resto}, Minimum number of drones:", round(m.ObjVal))

Restaurant : söder, Minimum number of drones: 10
Restaurant : hamngatan, Minimum number of drones: 9
Restaurant : hötorget, Minimum number of drones: 3
Restaurant : odenplan, Minimum number of drones: 6
Restaurant : kungsholmen, Minimum number of drones: 8
